In [30]:
#!pip install pandas plotly tabulate termcolor

In [31]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from tabulate import tabulate
from termcolor import colored

# DashBoards
from plotly.subplots import make_subplots


df_business=pd.read_csv('./dataset/Business Unit.csv')
df_customer=pd.read_csv('./dataset/Customer.csv')
df_dates=pd.read_csv('./dataset/Dates.csv')
df_financial=pd.read_excel('./dataset/Financial Sample.xlsx')

# Small Fixes 
df_financial.rename(columns={' Sales': 'Sales'}, inplace=True)
df_financial['Discount Band']=df_financial['Discount Band'].fillna('None') # Níveis de Desconto None, Low, Medium and HIGH
df_financial['Units Sold'] = df_financial['Units Sold'].astype('int64')

# Cria os Trimestres em Financial
df_financial['Dates_Quarter'] = df_financial['Month Number'].map({
    1: 'Q1', 2: 'Q1', 3: 'Q1',
    4: 'Q2', 5: 'Q2', 6: 'Q2',
    7: 'Q3', 8: 'Q3', 9: 'Q3',
    10: 'Q4', 11: 'Q4', 12: 'Q4'
})

In [32]:

def dataset_info():
      def print_dataframe_info(df, name):
            print(colored(str(f"{name}"),'green')+" contem " +
                  colored(str(f"{df.shape[0]}"),'red')+" linhas e "+
                  colored(str(f"{df.shape[1]}"),'red')+" colunas")
            print(colored(str("Sobre os Data Sets:\n"),'yellow'))
            print(tabulate(df.head(3), headers='keys', tablefmt='pretty')+"\n")

      print_dataframe_info(df_business, "Business")
      print_dataframe_info(df_customer, "Customer")
      print_dataframe_info(df_dates, "Dates")
      print_dataframe_info(df_financial, "Financial")

def financial_details():
      print(colored("Valores em Finacial:\n","light_magenta"))

      print('Discounts '+
            colored("["+str(f"{df_financial['Discounts'].min():.2f}, {df_financial['Discounts'].mean():.2f}, {df_financial['Discounts'].max():.2f}"),"green")+"]")
      print('Discount Band '+
            colored(str(f"[{', '.join(str(val) for val in df_financial['Discount Band'].unique())}]"),"green"))
      print('Product '+ 
            colored(str(f"[{', '.join(str(val) for val in df_financial['Product'].unique())}]"),"green"))
      print('Segment '+ 
            colored(str(f"[{', '.join(str(val) for val in df_financial['Segment'].unique())}]"),"green"))
      print('Country '+ 
            colored(str(f"[{', '.join(str(val) for val in df_financial['Country'].unique())}]"),"green"))

def columns_info():
      def print_column_types(df, name):
            column_types = pd.DataFrame(df.dtypes, columns=['Type'])
            column_types.index.name = 'Column Name'
            print(colored(f"\t\t{name}",'green'))
            print(tabulate(column_types, headers='keys', tablefmt='pretty')+"\n")
            
      print_column_types(df_business, "Business")
      print_column_types(df_customer, "Customer")
      print_column_types(df_dates, "Dates")
      print_column_types(df_financial, "Financial")

In [33]:
dataset_info()
financial_details()
columns_info()

Business contem 42 linhas e 2 colunas
Sobre os Data Sets:

+---+------------------------------------+----------------------------+
|   | Business_20_Units_Business_20_Unit | Business_20_Units_Division |
+---+------------------------------------+----------------------------+
| 0 |                23-0                |           Minor            |
| 1 |                50-0                |           Minor            |
| 2 |                55-0                |           Minor            |
+---+------------------------------------+----------------------------+

Customer contem 312 linhas e 5 colunas
Sobre os Data Sets:

+---+-----------------------------+----------------+----------------+--------------------------+-------------------------+
|   | Customers_Country_2f_Region | Customers_Name | Customers_City | Customers_Postal_20_Code | Customers_State_20_Code |
+---+-----------------------------+----------------+----------------+--------------------------+-------------------------+
| 0 |  

In [34]:
# Generale means "No Segmented"
# Global means "No Localized"

# Gráficos de Cartão

In [35]:
# Soma de Vendas por Produto Generale Global
df_sum_product_sales = (
    df_financial.groupby('Product')['Sales'].sum().sort_values(ascending=False).reset_index()
    .assign(formatted_sales= lambda x: x['Sales'].apply(lambda x: f"{(x / 1_000_000):.2f} Mi"))
)

fig_sum_product_sales = px.pie(df_sum_product_sales, 
             names='Product',      
             values='Sales',          
             title='Vendas por Produto - Global',
             hole=0,
            ).update_traces(
                 text=df_sum_product_sales['formatted_sales'],
                 textinfo='percent+text', textposition='inside'
            ).update_layout(width=600,height=400).show()

In [36]:
# Média de Sale Price por Product Generale Global
df_media_sale_price = df_financial.groupby('Product')['Sale Price'].mean().sort_values(ascending=False).reset_index() 

fig_media_sale_price = px.area(df_media_sale_price, 
              x='Product',        
              y='Sale Price',     
              title='Média de Preço de Vendas por Produto - Global',  
              markers=True,
              labels={'Product':'Produto','Sale Price':'Preço de Venda'}
              ).update_layout(width=600,height=400).show()


In [37]:
#Soma das Vendas por Trimestre e Segmento Global
df_sales_tri_segment = df_financial.groupby(['Segment', 'Dates_Quarter'])['Sales'].sum().sort_values(ascending=False).reset_index()

fig_sales_tri_segment = px.bar(
    df_sales_tri_segment,  
    x='Dates_Quarter',   
    y='Sales',           
    color='Segment',      
    barmode='group',     
    title='Soma das Vendas por Trimestre e Segmento',
    category_orders={'Dates_Quarter': ['Q1', 'Q2', 'Q3', 'Q4']}
).update_layout(width=600,height=400).show()

In [38]:
# Soma de vendas e Soma de Unidades Vendidas
print("Total Sales: "+f"{df_financial['Sales'].sum() / 1_000_000:.2f} Mi")
print("Units Sold: " + f"{df_financial['Units Sold'].sum() / 1_000_000:.2f} Mi")

# Gráficos de Cartão



Total Sales: 118.73 Mi
Units Sold: 1.13 Mi


In [39]:
# Soma de Profit por Country Genrale
df_sum_profit_country = (
    df_financial.groupby('Country')['Profit'].sum().sort_values(ascending=False).reset_index()
    .assign(formatted_sum_profit = lambda x: x['Profit'].apply(lambda x: f"{(x / 1_000_000):.2f} Mi"))
)
fig_sum_profit_country = px.pie(df_sum_profit_country, 
             names='Country',      
             values='Profit',          
             title='Lucro por País',
             hole=0).update_traces(
   text=df_sum_profit_country['formatted_sum_profit'],
   textinfo='percent+text', textposition='inside').update_layout(width=600,height=400).show()

In [40]:
# Média de profit por Ano Mês Generale Global
df_mean_profit_month = (
    df_financial.groupby(['Year','Month Name', 'Month Number'])['Profit'].sum().reset_index()
    .assign(month_year=lambda x: x['Month Name'] + ' ' + x['Year'].astype(str))    
    .sort_values(by=['Year','Month Number']).reset_index(drop=True)
)
fig_mean_profit_month = px.histogram(df_mean_profit_month, 
             x='month_year',      
             y='Profit',          
             title='Lucro por Mês/Ano').update_layout(width=600,height=400).show()

In [41]:
#Soma da Sales por Country Generale
df_sum_sales_country = (
    df_financial.groupby(['Country'])['Sales'].sum().reset_index()
)
fig_sum_sales_country = px.bar(df_sum_sales_country, 
             x='Country',      
             y='Sales',       
             title='Vendas por País').update_layout(width=800,height=400).show()

In [42]:
df_sum_profit_segment = (
    df_financial.groupby('Segment').sum('Profit').reset_index()
    .assign(formated_profit_sum=lambda x: x['Profit'].apply(lambda x: f"{(x / 1_000_000):.2f} Mi")).reset_index(drop=True)
)

fig_sum_profit_segment = px.pie(df_sum_profit_segment, 
             names='Segment',      
             values='Profit',          
             title='Lucro por País',
             hole=0).update_traces(
   text=df_sum_profit_segment['formated_profit_sum'],
   textinfo='percent+text', textposition='inside').update_layout(width=600,height=400).show()


In [43]:
df_sum_product_country_unit = df_financial.groupby(['Product', 'Country'])['Units Sold'].sum().sort_values(ascending=False).reset_index()

fig_sum_product_country_unit = px.scatter_geo(
    df_sum_product_country_unit,
    locations='Country',  
    locationmode='country names', 
    size='Units Sold',  
    color='Product', 
    hover_name='Country', 
    title='Mapa de Vendas Produto por País'
).update_layout(width=800,height=600).show()

In [44]:
# Valores Negativos em SUMProfit indicam Prejuízo  LOGO .abs() é ilógico
#OU os dados para um segmento estão errados
#print(df_financial.groupby(['Segment', 'Country'])['Profit'].sum())
#Enterprise        Canada                      -121508.750
#                  France                       -95749.375
#                  Germany                     -101473.750
#                  Mexico                      -120678.750
#                  United States of America    -175135.000

df_sum_profit_segment_country = df_financial.groupby(['Segment', 'Country'])['Profit'].sum().abs().reset_index()

fig_sum_profit_segment_country = px.scatter_geo(
    df_sum_profit_segment_country,
    locations='Country',  
    locationmode='country names', 
    size='Profit',  
    color='Segment', 
    hover_name='Country', 
    title='Mapa de Lucro por Segment/País'
).update_layout(width=800,height=600).show()

In [45]:
#Faltam alguns Controles, Country, Ano_Trimestre

fig_relatorio_vendas_segmento_tri = make_subplots(
    rows=2, cols=2,                  
    subplot_titles=('Vendas por Produto', 
                    'Média de Preço de Vendas por Produto', 
                    'Soma de Sales por Ano, Trimestre e Segmento'),
    specs=[[{'type': 'pie'}, {'type': 'histogram'}],
        [{'type': 'bar','colspan':2}, None]]
)

for trace in fig_sum_product_sales.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=1, col=1)

for trace in fig_media_sale_price.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=1, col=2)

for trace in fig_sales_tri_segment.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=2, col=1)

fig_relatorio_vendas_segmento_tri.update_layout(
    title_text="Relatório de Vendas por Segmento e Trimestre",
    showlegend=True,
    width=1400,  
    height=800
).show()

AttributeError: 'NoneType' object has no attribute 'data'

In [ ]:
# Relatório de Vendas por País e Lucro
fig_relatorio_vendas_segmento_tri = make_subplots(
    rows=2, cols=2,                  
    subplot_titles=('Soma de Profit por Country', 
                    'Média de Profit por Ano/Mês', 
                    'Soma de Sales por Country'),
    specs=[[{'type': 'pie'}, {'type': 'histogram'}],
        [{'type': 'bar', 'colspan':2}, None]]
)

# Adicionando os gráficos aos subgráficos
for trace in fig_sum_profit_country.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=1, col=1)

for trace in fig_mean_profit_month.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=1, col=2)

for trace in fig_sum_sales_country.data:
    fig_relatorio_vendas_segmento_tri.add_trace(trace, row=2, col=1)

# Atualizando o layout
fig_relatorio_vendas_segmento_tri.update_layout(
    title_text="Relatório de Vendas e Lucros",
    showlegend=True,
    width=1200,  
    height=800
).show()

In [37]:
# Relatório Distruição de Lucros, vendas e unidades por país e Segmento
fig_relatorio_profits_sales_country_segment = make_subplots(
    rows=2, cols=2,      
    subplot_titles=('Soma de Profit por Segment', 
                    'Soma de Units Sales, Segment e Country', 
                    'Soma de Profit Segment e Country'),
    specs=[[{'type': 'pie'}, {'type': 'scattergeo'}],[{'type': 'scattergeo', 'colspan':2},None]]
)

# Adicionando os gráficos aos subgráficos
for trace in fig_sum_profit_segment.data:
    fig_relatorio_profits_sales_country_segment.add_trace(trace, row=1, col=1)

for trace in fig_sum_product_country_unit.data:
    fig_relatorio_profits_sales_country_segment.add_trace(trace, row=1, col=2)

for trace in fig_sum_profit_segment_country.data:
    fig_relatorio_profits_sales_country_segment.add_trace(trace, row=2, col=1)

# Atualizando o layout
fig_relatorio_profits_sales_country_segment.update_layout(
    title_text="Distribuição de Lucros, vendas e Unidades por País e Segmento",
    showlegend=True,
    width=1400,  
    height=800
).show()


In [38]:
def export_graphics():
    fig_relatorio_vendas_segmento_tri.write_image("fig/fig_pag1.jpg", format="jpg")
    fig_relatorio_vendas_segmento_tri.write_image("fig/fig_pag2.jpg", format="jpg")
    fig_relatorio_profits_sales_country_segment.write_image("fig/fig_pag3.jpg", format="jpg")


# export_graphics()    

In [125]:
print("Total Sales: "+f"{df_financial['Sales'].sum() / 1_000_000:.2f} Mi")
print("Total de Unidades Vendidas: " + f"{df_financial['Units Sold'].sum() / 1_000_000:.2f} Mi")
print("Total de Descontos: -"+f"{df_financial['Discounts'].sum()/1_000_000:.2f} Mi")
print("Total de Gross Sales: "+f"{df_financial['Gross Sales'].sum()/1_000_000:.2f} Mi")
print("Total de COGS: "+f"{df_financial['COGS'].sum()/1_000_000:.2f} Mi")


ValueError: Invalid property specified for object of type plotly.graph_objs.indicator.Delta: 'color'

Did you mean "font"?

    Valid properties:
        decreasing
            :class:`plotly.graph_objects.indicator.delta.Decreasing
            ` instance or dict with compatible properties
        font
            Set the font used to display the delta
        increasing
            :class:`plotly.graph_objects.indicator.delta.Increasing
            ` instance or dict with compatible properties
        position
            Sets the position of delta with respect to the number.
        prefix
            Sets a prefix appearing before the delta.
        reference
            Sets the reference value to compute the delta. By
            default, it is set to the current value.
        relative
            Show relative change
        suffix
            Sets a suffix appearing next to the delta.
        valueformat
            Sets the value formatting rule using d3 formatting
           

In [ ]:
df_sales_segment = df_financial.groupby(['Segment'])['Sales'].sum().sort_values(ascending=True).reset_index()
df_sales_segment['formatted_sales'] = df_sales_segment['Sales'].apply(lambda x: f"${x/1_000_00:.2f} Mi")

fig_sales_segment_pie = px.pie(df_sales_segment, 
             names='Segment',      
             values='Sales',          
             title='Vendas por Segmento - Global',
             hole=0,
            ).update_traces(
                 text=df_sales_segment['formatted_sales'],
                 textinfo='percent+text', textposition='inside'
            ).update_layout(width=600,height=400).show()

fig_sales_segment_bar = px.bar(df_sales_segment, 
             y='Segment',      
             x='Sales',
             color='Sales',
             color_continuous_scale='Plasma',            
             title='Vendas por Segmento - Global'
            ).update_layout(width=600,height=400).show()

In [67]:
df_sales_product = df_financial.groupby(['Product'])['Sales'].sum().sort_values(ascending=True).reset_index()
df_sales_product['formatted_sales'] = df_sales_product['Sales'].apply(lambda x: f"${x/1_000_00:.2f} Mi")

fig_sales_product_pie = px.pie(df_sales_product, 
             names='Product',      
             values='Sales',          
             title='Vendas por Produto - Global',
             hole=0,
            ).update_traces(
                 text=df_sales_product['formatted_sales'],
                 textinfo='percent+text', textposition='inside'
            ).update_layout(width=600,height=400).show()

fig_sales_product_bar = px.bar(df_sales_product, 
             y='Product',      
             x='Sales',
             color='Sales',
             color_continuous_scale='Plasma',            
             title='Vendas por Produto - Global'
            ).update_layout(width=600,height=400).show()

In [85]:
df_sales_tri_segment = df_sales_tri_segment.groupby('Dates_Quarter')['Sales'].sum().reset_index()
df_sales_tri_segment['formatted_sales'] = df_sales_tri_segment['Sales'].apply(lambda x: f"${x/1_000_00:.2f} Mi")

go.Figure(
    go.Waterfall(
        x=df_sales_tri_segment['Dates_Quarter'],
        y=df_sales_tri_segment['Sales'],
        measure=["relative"] * (len(df_sales_tri_segment) - 1) + ["total"], 
        textposition="outside",
        hovertext=df_sales_tri_segment['formatted_sales'],  # Exibe formatted_sales no hover
        hoverinfo="text",  
        decreasing={"marker": {"color": "red"}}, 
        increasing={"marker": {"color": "green"}},  
        totals={"marker": {"color": "blue"}}
    )
).update_layout(title="Vendas por Trimestre").show()


In [123]:
df_sum_sales_units_products = df_financial.groupby("Product")['Units Sold'].sum().astype(int).reset_index()
df_sum_sales_units_products['formatted_units_sold'] = df_sum_sales_units_products['Units Sold'].apply(lambda x: f"{x/1_000_00:.2f} Mi")

fig = px.pie(df_sum_sales_units_products, names='Product', values='Units Sold', title='Unidades Vendidas por Produto', hole=0.3,
    custom_data=['formatted_units_sold'] ).update_traces(
    hovertemplate="<b>%{label}</b><br>Unidades Vendidas: %{customdata[0]}"
).show()

     Product  Units Sold formatted_units_sold
0   Amarilla      155315              1.55 Mi
1  Carretera      146846              1.47 Mi


In [122]:
fig_sum_profit_sum_discounts= go.Figure().add_trace(go.Scatter(
    x=df_sum_profit_sum_discounts['Year-Month'],
    y=df_sum_profit_sum_discounts['Profit'],
    hovertemplate='<b>Mês: %{x}</b><br>Lucro: %{customdata}<extra></extra>',
    fill='tozeroy',
    mode='lines',
    name='Lucro',
    customdata=df_sum_profit_sum_discounts['formatted_profit']
)
).add_trace(go.Scatter(
    x=df_sum_profit_sum_discounts['Year-Month'],
    y=df_sum_profit_sum_discounts['Discounts'],
    hovertemplate='<b>Mês: %{x}</b><br>Desconto: %{customdata}<extra></extra>',
    fill='tozeroy',
    mode='lines',
    name='Desconto',
    customdata=df_sum_profit_sum_discounts['formatted_discounts'] 
)
).update_layout(
    title='Lucro e Descontos por Mês',
    xaxis_title='Mês',
    yaxis_title='Valores',
    showlegend=True
).show()


In [101]:
# Agrupar os dados por produto e calcular o total de unidades vendidas
df_sum_sales_units_products = df_financial.groupby("Product")['Units Sold'].sum().astype(int).reset_index()

# Calcular a proporção das unidades vendidas por produto
df_sum_sales_units_products['percentage'] = (df_sum_sales_units_products['Units Sold'] / df_sum_sales_units_products['Units Sold'].sum()) * 100

# Adicionar texto formatado para o hover
df_sum_sales_units_products['hover_text'] = df_sum_sales_units_products.apply(
    lambda row: f"Produto: {row['Product']}<br>Unidades Vendidas: {row['Units Sold']}<br>Porcentagem: {row['percentage']:.2f}%", axis=1
)

# Definir os valores e categorias para o gráfico radar
values = df_sum_sales_units_products['percentage'].tolist() + [df_sum_sales_units_products['percentage'].iloc[0]]
categories = df_sum_sales_units_products['Product'].tolist() + [df_sum_sales_units_products['Product'].iloc[0]]
hover_text = df_sum_sales_units_products['hover_text'].tolist() + [df_sum_sales_units_products['hover_text'].iloc[0]]

# Criar o gráfico radar com hover text
fig_radar_product_units_sold = go.Figure(
    go.Scatterpolar(
        r=values,
        theta=categories,
        fill='toself',
        name='Proporção das Unidades Vendidas',
        hovertext=hover_text,  # Texto personalizado para hover
        hoverinfo='text'  # Exibir apenas o texto do hover
    )
).update_layout(
    title='Proporção das Unidades Vendidas por Produto',
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, max(values) * 1.1],
            title="Porcentagem (%)"
        )
    ),
    showlegend=False
).show()

